<a href="https://colab.research.google.com/github/Zafar488/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zafar488/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip install -q duckdb huggingface_hub

In [3]:
import os
import duckdb
import numpy as np
import pandas as pd

from google.colab import userdata
from huggingface_hub import whoami

# Safely load Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN nahi mila. Colab Secrets panel mein HF_TOKEN add karo."
    )

user_info = whoami(token=HF_TOKEN)
print("Hugging Face login successful:", user_info["name"])

con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{safe_token}'
    )
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

# Mid-panel month only
MARCH_FACT = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

print("DuckDB connection ready.")
print("Development month: March 2026")

Hugging Face login successful: zafar4050
DuckDB connection ready.
Development month: March 2026


## 1. Unit of Analysis and Time Window

### One row

One row in my final feature frame represents one anonymized content page for one client, aggregated over March 2026.

The source table has one row per content page per day. I aggregate those daily records into one page-level row because my lane ranks pages for human content review.

### Tables

I use `fact_content_daily_performance` from the FlyRank internship warehouse.

The daily table provides observed search-performance measurements such as impressions, clicks, average position, and data-availability indicators.

### Time window

I use the mid-panel month March 2026.

The feature window is March 1–15, 2026. The provisional outcome window is March 16–31, 2026.

This keeps the feature period before the outcome period and avoids using June 2026, which is being treated as a sealed final test month.

### What I predict or rank

My provisional proxy identifies pages whose average daily impressions in the outcome window are more than 20% below their average daily impressions in the feature window.

The final business output is a ranked review queue, not an automatic editing decision.

### Deliberately excluded

I exclude all outcome-window measurements from the honest feature set because they would only be known after the decision moment.

I also exclude anonymized identifiers as predictive features. They are used only for grouping and client-holdout validation.

In [4]:
contract = {
    "unit_of_analysis": (
        "One anonymized content page for one client, "
        "aggregated over March 2026"
    ),
    "source_table": "fact_content_daily_performance",
    "feature_window": "2026-03-01 to 2026-03-15",
    "outcome_window": "2026-03-16 to 2026-03-31",
    "ranking_goal": "Prioritize pages for human content review",
    "excluded": "Outcome-window measurements from honest features",
}

for key, value in contract.items():
    print(f"{key.replace('_', ' ').title()}: {value}")

Unit Of Analysis: One anonymized content page for one client, aggregated over March 2026
Source Table: fact_content_daily_performance
Feature Window: 2026-03-01 to 2026-03-15
Outcome Window: 2026-03-16 to 2026-03-31
Ranking Goal: Prioritize pages for human content review
Excluded: Outcome-window measurements from honest features


## 2. Field Roles

### Features — exactly five

1. `feature_impressions`  
   Knowable at the decision moment because it is calculated only from impressions measured during March 1–15.

2. `feature_clicks`  
   Knowable at the decision moment because it uses clicks observed only during the feature window.

3. `feature_avg_position`  
   Knowable at the decision moment because it summarizes search position before the outcome window begins.

4. `feature_active_days`  
   Knowable at the decision moment because it counts the number of feature-window days on which the page received impressions.

5. `feature_position_volatility`  
   Knowable at the decision moment because it measures variation in average position only during March 1–15.

### Label or proxy

`is_declining_proxy` equals 1 when the page's average daily impressions during March 16–31 are more than 20% below its average daily impressions during March 1–15. Otherwise, it equals 0.

This is a provisional future-window proxy. It does not prove that a page requires a refresh or that refreshing it will cause recovery.

### Context fields

- `client_hash_id`: used for grouped validation.
- `content_hash_id`: identifies the anonymized content page.
- `report_date`: defines the feature and outcome windows.
- `gsc_data_available`: confirms that Search Console data was available.

### Excluded fields

- Outcome-window impressions and clicks are excluded from the honest model because they occur after the decision moment.
- `decline_ratio` is excluded because the label is calculated directly from it.
- Client and content identifiers are excluded as model features because their scrambled values have no predictive meaning.
- GA4 engagement measurements are not used in this first slice because GA4 coverage begins at different times for different clients.
- The June 2026 sample is excluded from development because it is reserved as a later test period.

In [5]:
field_roles = pd.DataFrame(
    [
        ["feature_impressions", "feature", "Measured in March 1–15"],
        ["feature_clicks", "feature", "Measured in March 1–15"],
        ["feature_avg_position", "feature", "Measured in March 1–15"],
        ["feature_active_days", "feature", "Measured in March 1–15"],
        [
            "feature_position_volatility",
            "feature",
            "Measured in March 1–15",
        ],
        ["is_declining_proxy", "label/proxy", "Measured after feature window"],
        ["client_hash_id", "context", "Grouping and validation only"],
        ["content_hash_id", "context", "Page identifier only"],
        ["gsc_data_available", "context", "Availability check"],
        [
            "decline_ratio",
            "excluded",
            "Directly defines the label and causes leakage",
        ],
    ],
    columns=["field", "role", "reason"],
)

field_roles

,field,role,reason
0,feature_impressions,feature,Measured in March 1–15
1,feature_clicks,feature,Measured in March 1–15
2,feature_avg_position,feature,Measured in March 1–15
3,feature_active_days,feature,Measured in March 1–15
4,feature_position_volatility,feature,Measured in March 1–15
5,is_declining_proxy,label/proxy,Measured after feature window
6,client_hash_id,context,Grouping and validation only
7,content_hash_id,context,Page identifier only
8,gsc_data_available,context,Availability check
9,decline_ratio,excluded,Directly defines the label and causes leakage


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
# Verification Query 1: source grain

grain_check = con.sql(f"""
    WITH key_counts AS (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            COUNT(*) AS rows_per_key
        FROM {MARCH_FACT}
        GROUP BY
            report_date,
            client_hash_id,
            content_hash_id
    )
    SELECT
        SUM(rows_per_key) AS total_source_rows,
        COUNT(*) AS distinct_daily_keys,
        SUM(
            CASE
                WHEN rows_per_key > 1
                THEN rows_per_key - 1
                ELSE 0
            END
        ) AS duplicate_rows,
        MAX(rows_per_key) AS maximum_rows_per_daily_key
    FROM key_counts
""").df()

print("Verification Query 1 — Grain")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Verification Query 1 — Grain


,total_source_rows,distinct_daily_keys,duplicate_rows,maximum_rows_per_daily_key
0,9841378.0,9841378,0.0,1


In [7]:
# Verification Query 2: page population and date span

count_and_dates = con.sql(f"""
    SELECT
        COUNT(*) AS daily_rows,
        COUNT(
            DISTINCT (client_hash_id, content_hash_id)
        ) AS unique_content_pages,
        COUNT(DISTINCT client_hash_id) AS clients,
        MIN(report_date) AS minimum_date,
        MAX(report_date) AS maximum_date
    FROM {MARCH_FACT}
""").df()

print("Verification Query 2 — Count and Date Span")
count_and_dates

Verification Query 2 — Count and Date Span


,daily_rows,unique_content_pages,clients,minimum_date,maximum_date
0,9841378,331437,55,2026-03-01,2026-03-31


In [8]:
# Verification Query 3: GSC availability

availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS all_march_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS rows_with_gsc_available,
        COUNT(
            DISTINCT (client_hash_id, content_hash_id)
        ) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS pages_with_gsc_available,
        ROUND(
            100.0
            * COUNT(*) FILTER (
                WHERE gsc_data_available IS TRUE
            )
            / COUNT(*),
            2
        ) AS available_row_percentage
    FROM {MARCH_FACT}
""").df()

print("Verification Query 3 — Availability")
availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Verification Query 3 — Availability


,all_march_rows,rows_with_gsc_available,pages_with_gsc_available,available_row_percentage
0,9841378,3611061,176738,36.69


In [9]:
feature_frame = con.sql(f"""
    WITH page_windows AS (
        SELECT
            client_hash_id,
            content_hash_id,

            -- Feature 1
            SUM(
                CASE
                    WHEN report_date BETWEEN
                         DATE '2026-03-01'
                         AND DATE '2026-03-15'
                     AND gsc_data_available IS TRUE
                    THEN COALESCE(gsc_impressions, 0)
                    ELSE 0
                END
            ) AS feature_impressions,

            -- Feature 2
            SUM(
                CASE
                    WHEN report_date BETWEEN
                         DATE '2026-03-01'
                         AND DATE '2026-03-15'
                     AND gsc_data_available IS TRUE
                    THEN COALESCE(gsc_clicks, 0)
                    ELSE 0
                END
            ) AS feature_clicks,

            -- Feature 3
            AVG(
                CASE
                    WHEN report_date BETWEEN
                         DATE '2026-03-01'
                         AND DATE '2026-03-15'
                     AND gsc_data_available IS TRUE
                    THEN gsc_avg_position
                END
            ) AS feature_avg_position,

            -- Feature 4
            COUNT(
                DISTINCT CASE
                    WHEN report_date BETWEEN
                         DATE '2026-03-01'
                         AND DATE '2026-03-15'
                     AND gsc_data_available IS TRUE
                     AND COALESCE(gsc_impressions, 0) > 0
                    THEN report_date
                END
            ) AS feature_active_days,

            -- Feature 5
            STDDEV_SAMP(
                CASE
                    WHEN report_date BETWEEN
                         DATE '2026-03-01'
                         AND DATE '2026-03-15'
                     AND gsc_data_available IS TRUE
                    THEN gsc_avg_position
                END
            ) AS feature_position_volatility,

            -- Outcome information: not honest model features
            SUM(
                CASE
                    WHEN report_date BETWEEN
                         DATE '2026-03-16'
                         AND DATE '2026-03-31'
                     AND gsc_data_available IS TRUE
                    THEN COALESCE(gsc_impressions, 0)
                    ELSE 0
                END
            ) AS outcome_impressions,

            COUNT(
                DISTINCT CASE
                    WHEN report_date BETWEEN
                         DATE '2026-03-16'
                         AND DATE '2026-03-31'
                     AND gsc_data_available IS TRUE
                    THEN report_date
                END
            ) AS outcome_available_days

        FROM {MARCH_FACT}
        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT *
    FROM page_windows
    WHERE feature_impressions >= 100
      AND feature_active_days >= 5
      AND outcome_available_days >= 5
""").df()

# Daily averages make unequal window lengths comparable
feature_frame["feature_daily_impressions"] = (
    feature_frame["feature_impressions"] / 15
)

feature_frame["outcome_daily_impressions"] = (
    feature_frame["outcome_impressions"]
    / feature_frame["outcome_available_days"]
)

feature_frame["is_declining_proxy"] = (
    feature_frame["outcome_daily_impressions"]
    < 0.80 * feature_frame["feature_daily_impressions"]
).astype(int)

feature_columns = [
    "feature_impressions",
    "feature_clicks",
    "feature_avg_position",
    "feature_active_days",
    "feature_position_volatility",
]

print("Feature-frame rows:", f"{len(feature_frame):,}")
print(
    "Declining proxy rate:",
    round(feature_frame["is_declining_proxy"].mean(), 3),
)

feature_frame[
    [
        "client_hash_id",
        "content_hash_id",
        *feature_columns,
        "is_declining_proxy",
    ]
].head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-frame rows: 76,201
Declining proxy rate: 0.323


,client_hash_id,content_hash_id,feature_impressions,feature_clicks,feature_avg_position,feature_active_days,feature_position_volatility,is_declining_proxy
0,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,199.0,2.0,4.086084,13,0.560314,0
1,client_62f4a7e64f5e0096,content_275b6f7f733016d4,467.0,1.0,4.449176,13,1.003601,1
2,client_62f4a7e64f5e0096,content_755d951187fcd70a,771.0,1.0,1.883472,14,0.777534,0
3,client_62f4a7e64f5e0096,content_92c381fbd361212e,233.0,0.0,4.807270,13,1.887822,0
4,client_62f4a7e64f5e0096,content_97188a7032a705cf,230.0,2.0,3.760644,13,1.622677,0


In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
)

model_df = feature_frame.dropna(
    subset=feature_columns
    + ["client_hash_id", "is_declining_proxy"]
).copy()

X = model_df[feature_columns]
y = model_df["is_declining_proxy"]
groups = model_df["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42,
)

train_index, test_index = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_index]
X_test = X.iloc[test_index]

y_train = y.iloc[train_index]
y_test = y.iloc[test_index]

train_clients = set(groups.iloc[train_index])
test_clients = set(groups.iloc[test_index])

honest_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

honest_model.fit(X_train, y_train)

honest_probability = honest_model.predict_proba(X_test)[:, 1]
honest_prediction = honest_model.predict(X_test)

honest_accuracy = accuracy_score(
    y_test,
    honest_prediction,
)

honest_auc = roc_auc_score(
    y_test,
    honest_probability,
)

print("HONEST MODEL")
print("-" * 45)
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print(
    "Client overlap:",
    len(train_clients.intersection(test_clients)),
)
print(f"Honest accuracy: {honest_accuracy:.3f}")
print(f"Honest ROC AUC: {honest_auc:.3f}")
print()
print(
    classification_report(
        y_test,
        honest_prediction,
        digits=3,
        zero_division=0,
    )
)

HONEST MODEL
---------------------------------------------
Train clients: 27
Test clients: 9
Client overlap: 0
Honest accuracy: 0.550
Honest ROC AUC: 0.619

              precision    recall  f1-score   support

           0      0.773     0.479     0.591     17189
           1      0.389     0.702     0.501      8123

    accuracy                          0.550     25312
   macro avg      0.581     0.590     0.546     25312
weighted avg      0.650     0.550     0.562     25312



In [11]:
model_df["decline_ratio"] = (
    model_df["outcome_daily_impressions"]
    / model_df["feature_daily_impressions"]
)

In [12]:
# Deliberate leakage experiment

model_df["decline_ratio"] = (
    model_df["outcome_daily_impressions"]
    / model_df["feature_daily_impressions"]
)

leaky_feature_columns = feature_columns + ["decline_ratio"]

X_leaky = model_df[leaky_feature_columns]

X_leaky_train = X_leaky.iloc[train_index]
X_leaky_test = X_leaky.iloc[test_index]

leaky_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=4,
    random_state=42,
    n_jobs=-1,
)

leaky_model.fit(X_leaky_train, y_train)

leaky_probability = leaky_model.predict_proba(
    X_leaky_test
)[:, 1]

leaky_prediction = leaky_model.predict(
    X_leaky_test
)

leaky_accuracy = accuracy_score(
    y_test,
    leaky_prediction,
)

leaky_auc = roc_auc_score(
    y_test,
    leaky_probability,
)

print("DELIBERATE LEAKAGE EXPERIMENT")
print("-" * 45)
print(f"Honest accuracy: {honest_accuracy:.3f}")
print(f"Leaky accuracy:  {leaky_accuracy:.3f}")
print(f"Honest ROC AUC:  {honest_auc:.3f}")
print(f"Leaky ROC AUC:   {leaky_auc:.3f}")
print()
print(
    "The score jumped because decline_ratio "
    "contains the answer used to define the label."
)

# Remove the leaking feature permanently
model_df.drop(
    columns=["decline_ratio"],
    inplace=True,
)

assert "decline_ratio" not in model_df.columns

print("\ndecline_ratio removed.")
print("Final feature set:", feature_columns)
print(f"Final retained honest ROC AUC: {honest_auc:.3f}")

DELIBERATE LEAKAGE EXPERIMENT
---------------------------------------------
Honest accuracy: 0.550
Leaky accuracy:  1.000
Honest ROC AUC:  0.619
Leaky ROC AUC:   1.000

The score jumped because decline_ratio contains the answer used to define the label.

decline_ratio removed.
Final feature set: ['feature_impressions', 'feature_clicks', 'feature_avg_position', 'feature_active_days', 'feature_position_volatility']
Final retained honest ROC AUC: 0.619


### Leakage Experiment Observation

I deliberately added `decline_ratio` as a feature. This value compares outcome-window impressions with feature-window impressions, and the proxy label is directly defined from the same ratio.

The model's score increased sharply because the feature contained the answer in disguised form. This was data leakage, not genuine predictive skill.

I removed `decline_ratio` from the final feature set and retained the honest grouped-validation score.

The final model uses only the five measurements that were available before the outcome window began.

## 4. Data Limits

### Named limitation: unbalanced client history

The warehouse is an unbalanced panel. Different clients began GSC and GA4 tracking on different dates, so they do not all have the same amount of historical coverage.

Some clients are GSC-only, while others have both GSC and GA4 measurements. Missing GA4 values must therefore not automatically be interpreted as zero engagement or zero traffic.

This analysis uses only March 2026, so it cannot fully represent seasonality or long-term content behaviour.

The provisional decline label measures an observed reduction in impressions. It does not prove why the decline occurred, that a refresh is necessary, or that editing a page would cause recovery.

The results are directional and intended to support human review.

In [13]:
limitations_summary = {
    "development_month": "March 2026 only",
    "panel_structure": "Clients have different tracking start dates",
    "availability": "Some clients are GSC-only",
    "causality": "Observed decline does not prove refresh impact",
    "intended_use": "Human decision-support",
}

for name, description in limitations_summary.items():
    print(f"{name.replace('_', ' ').title()}: {description}")

Development Month: March 2026 only
Panel Structure: Clients have different tracking start dates
Availability: Some clients are GSC-only
Causality: Observed decline does not prove refresh impact
Intended Use: Human decision-support


## Results Summary

The March 2026 source table contained 9,841,378 daily rows covering 331,437 anonymized content pages across 55 clients. The grain check found no duplicate client-content-date keys.

Only 36.69% of March rows had GSC data available, which shows that availability must be checked explicitly and missing measurements must not automatically be interpreted as zero performance.

The final page-level feature frame contained 76,201 pages and used exactly five features measured before the outcome window. The declining proxy rate was 32.3%.

Using GroupShuffleSplit, the honest model was tested on unseen clients with zero client overlap. It achieved an accuracy of 0.550 and a ROC AUC of 0.619. This suggests that the five pre-outcome features carry some directional signal, but the result is not strong enough to support automatic decisions.

When I deliberately added the label-derived decline_ratio feature, both accuracy and ROC AUC increased to 1.000. This was data leakage rather than real predictive skill. I removed the leaking feature and retained the honest ROC AUC of 0.619.

The analysis is intended for human decision-support and does not prove that refreshing a page will cause recovery.

## 5. Self-check

- [x] I gave five plain-language data-contract answers.
- [x] I defined one page-level unit of analysis.
- [x] I used March 2026 as a mid-panel development month.
- [x] I included exactly three verification queries.
- [x] I checked availability using `IS TRUE`.
- [x] I built exactly five honest features.
- [x] I explained when every feature was available.
- [x] I deliberately demonstrated one leakage trap.
- [x] I deleted the leaking feature.
- [x] I retained the honest grouped-validation result.
- [x] I named the unbalanced-panel limitation.
- [x] I used observational and decision-support language.
- [x] I included no client names, domains, URLs, or private queries.